# Time in Zone: Market Square Fountain

Zone counting tells us *how many* people are in an area right now. This notebook answers *how long* each person stays there, which separates people who linger at the fountain from people who just walk past.

It combines the pieces from the previous notebooks:

1. [RF-DETR](https://github.com/roboflow/rf-detr) with tiled inference to detect people in a 4K frame.
2. ByteTrack from [trackers](https://github.com/roboflow/trackers) so each person keeps the same ID across frames.
3. `sv.PolygonZone` to check who is inside the fountain area on each frame.
4. A simple per-ID timer: count the frames each tracked person spends in the zone and convert them to seconds.

## Setup

In [ ]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import supervision as sv
from rfdetr import RFDETRNano
from rfdetr.assets.coco_classes import COCO_CLASSES
from supervision.assets import VideoAssets, download_assets
from tqdm.auto import tqdm
from trackers import ByteTrackTracker

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

## Download the video

The same fixed-camera clip as the zone-counting notebook: portrait 4K at 60 fps, about 8 seconds long.

In [ ]:
SOURCE_VIDEO_PATH = download_assets(VideoAssets.MARKET_SQUARE, directory=DATA_DIR)
TARGET_VIDEO_PATH = OUTPUT_DIR / "market_square_time_in_zone.mp4"

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
video_info

## Detector, tracker and zone

Same tiled detector as before. The tracker gets every detection down to 0.3 confidence: ByteTrack uses the low-confidence ones to keep existing tracks alive, but only starts new tracks from confident detections.

We process every third frame (20 fps effective), so the tracker is told the effective frame rate and times are computed with the same stride.

In [ ]:
model = RFDETRNano()
PERSON_CLASS_ID = next(k for k, v in COCO_CLASSES.items() if v == "person")


def detect_batch(images):
    return model.predict([image[:, :, ::-1].copy() for image in images], threshold=0.3)


slicer = sv.InferenceSlicer(
    callback=detect_batch,
    slice_wh=640,
    overlap_wh=128,
    batch_size=8,
    overlap_metric=sv.OverlapMetric.IOS,
)


def detect_people(frame):
    detections = slicer(frame)
    return detections[detections.class_id == PERSON_CLASS_ID]


STRIDE = 3
effective_fps = video_info.fps / STRIDE
tracker = ByteTrackTracker(frame_rate=effective_fps)

# The fountain plus the walkway around it, so we see both people who stop and people who pass by.
FOUNTAIN = np.array([(700, 860), (1500, 860), (1820, 1140), (1820, 1680), (1520, 1920), (680, 1920), (380, 1680), (380, 1140)])
zone = sv.PolygonZone(polygon=FOUNTAIN)

## The timer

Supervision doesn't ship a timer, but it's only a few lines: a dictionary from `tracker_id` to the number of processed frames that ID has spent inside the zone. Dividing by the effective frame rate gives seconds.

This measures **total** time in the zone. If someone leaves and comes back, both visits add up, as long as the tracker keeps the same ID.

In [ ]:
frames_in_zone = defaultdict(int)


def seconds_in_zone(tracker_id):
    return frames_in_zone[tracker_id] / effective_fps

## Annotation

People inside the zone get a coloured box and a label with their running time; everyone else is drawn as a thin grey box. The label colour moves from green to red as the time grows, so lingering people stand out.

In [ ]:
text_scale = sv.calculate_optimal_text_scale(video_info.resolution_wh)
thickness = sv.calculate_optimal_line_thickness(video_info.resolution_wh)

TIME_PALETTE = sv.ColorPalette.from_hex(["#22c55e", "#eab308", "#f97316", "#ef4444"])
TIME_STEPS_S = [2, 4, 6]  # thresholds between the palette colours


def time_color_index(detections):
    times = np.array([seconds_in_zone(tracker_id) for tracker_id in detections.tracker_id])
    return np.searchsorted(TIME_STEPS_S, times, side="right")


zone_annotator = sv.PolygonZoneAnnotator(
    zone=zone, color=sv.Color.from_hex("#3b82f6"), thickness=thickness, text_scale=text_scale,
    text_thickness=thickness, opacity=0.15, display_in_zone_count=False,
)
box_annotator = sv.BoxAnnotator(color=TIME_PALETTE, thickness=thickness, color_lookup=sv.ColorLookup.INDEX)
label_annotator = sv.LabelAnnotator(
    color=TIME_PALETTE, text_scale=text_scale, text_thickness=thickness, color_lookup=sv.ColorLookup.INDEX,
    smart_position=True,
)
outside_annotator = sv.BoxAnnotator(color=sv.Color.from_hex("#9ca3af"), thickness=max(1, thickness // 2))

In [ ]:
def process(frame):
    detections = detect_people(frame)
    detections = tracker.update(detections)
    detections = detections[detections.tracker_id != -1]

    in_zone = zone.trigger(detections)
    inside, outside = detections[in_zone], detections[~in_zone]
    for tracker_id in inside.tracker_id:
        frames_in_zone[tracker_id] += 1

    labels = [f"#{tracker_id} {seconds_in_zone(tracker_id):.1f}s" for tracker_id in inside.tracker_id]
    color_index = time_color_index(inside)

    annotated = zone_annotator.annotate(frame.copy())
    annotated = outside_annotator.annotate(annotated, outside)
    annotated = box_annotator.annotate(annotated, inside, custom_color_lookup=color_index)
    annotated = label_annotator.annotate(annotated, inside, labels, custom_color_lookup=color_index)
    return annotated

## Process the video

About a minute and a half on an Apple Silicon Mac.

In [ ]:
output_info = sv.VideoInfo(width=video_info.width, height=video_info.height, fps=effective_fps)
frames = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, stride=STRIDE)

with sv.VideoSink(str(TARGET_VIDEO_PATH), output_info) as sink:
    for frame in tqdm(frames, total=video_info.total_frames // STRIDE + 1):
        sink.write_frame(process(frame))

## Results

One row per tracked person who entered the fountain area, sorted by time spent.

In [ ]:
dwell = (
    pd.Series({tracker_id: seconds_in_zone(tracker_id) for tracker_id in frames_in_zone}, name="seconds")
    .rename_axis("tracker_id")
    .sort_values(ascending=False)
    .round(1)
)
print(f"{len(dwell)} people entered the fountain area")
dwell.to_frame().T

In [ ]:
clip_length_s = video_info.total_frames / video_info.fps
bins = np.arange(0, np.ceil(clip_length_s) + 1)
ax = dwell.plot.hist(bins=bins, figsize=(10, 4), color="#3b82f6", edgecolor="white")
ax.axvline(clip_length_s, color="#ef4444", linestyle="--", label=f"clip length ({clip_length_s:.1f}s)")
ax.set_xlabel("seconds in the fountain area")
ax.set_ylabel("people")
ax.set_title("Time spent at the fountain")
ax.legend();

In [ ]:
output_frames = video_info.total_frames // STRIDE
samples = list(sv.get_video_frames_generator(TARGET_VIDEO_PATH, stride=output_frames // 4))[1:4]
sv.plot_images_grid(samples, grid_size=(1, 3), size=(16, 10))

## Reading the results

Two groups usually show up: people **passing through** the edge of the zone for a second or two, and people **lingering** (sitting on the fountain's rim, waiting, taking photos) for most of the clip.

Keep in mind the limits of this simple approach:

- **Censored times.** Anyone already at the fountain when the clip starts, or still there when it ends, stays *at least* the measured time. With an 8-second clip, the long bar at the right is really "8 seconds or more".
- **ID switches.** If the tracker loses a person (occlusion, a missed detection) and gives them a new ID, their time is split across two rows. A larger `lost_track_buffer` on the tracker helps.
- **Zone edges.** Someone walking along the border can flicker in and out. Requiring a minimum time (e.g. ignore anyone under 1 s) filters those out.

Open `output/market_square_time_in_zone.mp4` to watch the timers run.

**Things to try**

- Report *continuous* time instead of total: reset an ID's counter when it leaves the zone.
- Add the café terrace as a second zone and compare the two.
- Apply this to a queue (checkout, ticket machine) to estimate waiting times.